# Lesson 0 — Prerequisites: *Environment & Data Setup*
### Practical Machine Learning for Transcriptomics in Cancer Research

**Run this notebook once, before Lessons 1–5.** It does two one-time setup steps:

1. **Creates the environment** — a dedicated **conda** environment named `ml26`, registered as a
   Jupyter kernel, with the scientific stack (NumPy, pandas, scikit-learn, matplotlib) plus the
   data-download helpers (`requests`, `GEOparse`). *(Section 1 — done in a terminal first.)*
2. **Downloads the data into a shared cache** — **METABRIC** (~660 MB, cBioPortal datahub) and
   **GSE6532** (~180 MB, NCBI GEO). *(Sections 2–4 — run the cells here.)*

> **Internet is required** for the download section. The first run takes a few minutes (the METABRIC
> expression matrix is large). After that everything is cached and offline-friendly. The cache is
> **git-ignored** — never commit it.


## Section 1 — Create the environment (conda)

Do this **once, in a terminal, *before* you open this notebook.** The steps create an isolated conda
environment named `ml26` and register it as a Jupyter kernel, so this notebook — and every lesson —
runs against the exact same packages.

> **Don't have conda?** Install **Miniconda** first (a minimal conda, ~5 minutes). Installers and
> per-OS instructions for macOS / Windows / Linux are here:
> **https://docs.anaconda.com/miniconda/**

From the **repository root**:

```bash
# 1) create an isolated environment with Python 3.11
conda create -n ml26 python=3.11 -y

# 2) activate it
conda activate ml26

# 3) install the course packages
pip install -r requirements.txt

# 4) register this environment as a Jupyter kernel named "ml26"
python -m ipykernel install --user --name ml26 --display-name "Python (ml26)"

# 5) launch Jupyter and open this notebook
jupyter lab
```

Then, **in Jupyter, select the `Python (ml26)` kernel** — use the kernel picker (top-right) or
*Kernel → Change Kernel*. Every notebook in this course is set to the `ml26` kernel. The cell below
confirms you are running inside it.


In [ ]:
# Confirm you are running inside the `ml26` environment, and top up any missing packages.
import sys, subprocess
from pathlib import Path

print("Python executable:", sys.executable)
print("(this should live inside .../envs/ml26/ if you selected the 'Python (ml26)' kernel)\n")

def _find_requirements():
    """Locate requirements.txt: prefer the lesson-00 copy, then the repo-root copy."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        for cand in [parent / "lessons" / "lesson00_prerequisites" / "requirements.txt",
                     parent / "requirements.txt"]:
            if cand.is_file():
                return cand
    return None

req = _find_requirements()
if req is not None:
    print(f"Installing/updating packages from {req} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
else:
    print("requirements.txt not found; install it manually with: pip install -r requirements.txt")
print("Environment check complete.")


In [ ]:
# Confirm the core stack imports, and report versions so setup problems surface here (not mid-lesson).
import numpy, pandas, sklearn, matplotlib, requests
print(f"{'numpy':13s}", numpy.__version__)
print(f"{'pandas':13s}", pandas.__version__)
print(f"{'scikit-learn':13s}", sklearn.__version__)
print(f"{'matplotlib':13s}", matplotlib.__version__)
print(f"{'requests':13s}", requests.__version__)
try:
    import GEOparse
    print(f"{'GEOparse':13s}", GEOparse.__version__)
except Exception as e:
    print(f"{'GEOparse':13s} NOT importable ->", e)
try:
    import lifelines
    print(f"{'lifelines':13s}", lifelines.__version__, "(optional; one Lesson 5 cell uses it)")
except Exception:
    print(f"{'lifelines':13s} not installed (optional — Lesson 5 skips its one cell automatically)")


## 📚 The libraries in this course — official docs

You just installed and version-checked the stack above. Here is the authoritative reference for each — bookmark the ones you will lean on. Every lesson notebook also carries a short, tailored version of this list in its own setup section.

**Core scientific stack**

| Library | What we use it for | Official docs |
|---|---|---|
| **NumPy** | arrays, vectorised math, RNG seeding | https://numpy.org/doc/stable/ |
| **pandas** | expression / clinical tables, joins, indexing | https://pandas.pydata.org/docs/ |
| **Matplotlib** | every plot in the course | https://matplotlib.org/stable/index.html |
| **scikit-learn** | pipelines, models, cross-validation, metrics | https://scikit-learn.org/stable/ |

**Data download & parsing**

| Library / source | Role | Link |
|---|---|---|
| **requests** | streams the METABRIC files over HTTPS | https://requests.readthedocs.io/en/latest/ |
| **GEOparse** | parses the GSE6532 SOFT file from NCBI GEO | https://geoparse.readthedocs.io/en/latest/ |
| **cBioPortal datahub** | source of the METABRIC (`brca_metabric`) cohort | https://github.com/cBioPortal/datahub/tree/master/public/brca_metabric |
| **NCBI GEO — GSE6532** | second (external) cohort, Loi et al. | https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE6532 |

**Optional (Lesson 5 only)**

| Library | Role | Link |
|---|---|---|
| **lifelines** | Kaplan–Meier / survival (time-to-event) analysis | https://lifelines.readthedocs.io/en/latest/ |

> **How to read the scikit-learn docs:** each tool has a **User Guide** page (the *why* and the intuition) and an **API reference** page (every parameter). When a lesson names a class such as `LogisticRegression`, searching that name jumps to its API page; the "User Guide" link at the top of it gives the narrative.


## Section 2 — Locate the shared data cache

The helper below resolves the **one** folder every lesson shares for cached data, by walking up to the repository root and anchoring on `lessons/lesson01_*/practical/task/datasets/`. This is the **exact same resolver the lesson notebooks use**, so whatever we download here is found by Lessons 1–5.


In [ ]:
# Core scientific stack (usually preinstalled). GEOparse fetches GSE6532 from GEO.
# Run once; safe to re-run.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import GEOparse  # noqa
except ImportError:
    _pip("GEOparse")

import os, tarfile, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
np.random.seed(0)          # reproducibility for the splits later

def _resolve_data_dir():
    """Find the lesson's practical/task/datasets/ folder regardless of where Jupyter
    was launched (the student notebook runs from notebooks/, the solution from
    solutions/lesson01_biological_question/notebooks/ — both should share ONE cache).

    Repo layout this targets:
      lessons/lesson01_biological_question/practical/task/
        ├── notebooks/   <- student notebook usually runs from here
        └── datasets/    <- data is cached here (git-ignored, kept via .gitkeep)
    """
    here = Path.cwd()
    # 1) PRIORITY: walk upward to the repo root, then locate the lesson-01 datasets anchor
    #    by glob (robust to the exact topic suffix), so the student and solution notebooks
    #    share the same cache.
    for parent in [here, *here.parents]:
        lessons = parent / "lessons"
        if lessons.is_dir():
            for cand in sorted(lessons.glob("lesson01_*/practical/task/datasets")):
                cand.mkdir(parents=True, exist_ok=True)
                return cand
            # lessons/ exists but no datasets dir yet -> create the canonical one if the
            # lesson folder is present
            for cand in sorted(lessons.glob("lesson01_*/practical/task")):
                d = cand / "datasets"; d.mkdir(parents=True, exist_ok=True)
                return d
    # 2) Otherwise, try datasets next to / beside the notebook.
    for c in [here.parent / "datasets", here / "datasets"]:
        if c.parent.exists():
            c.mkdir(parents=True, exist_ok=True)
            return c
    # 3) Last resort.
    fallback = here / "datasets"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback

DATA_DIR = str(_resolve_data_dir())
print("Setup complete. Data will be cached in:")
print("  ", os.path.abspath(DATA_DIR))
print("(Large downloads are git-ignored; the folder is kept via .gitkeep.)")

## Section 3 — Download the cohorts (METABRIC + GSE6532)

These are the **same loader functions** the lessons define. Running them here streams each file into the shared cache (skipping anything already present) and parses it to confirm the download is intact.

- **METABRIC** — Illumina microarray expression (~660 MB) + clinical tables, from the cBioPortal datahub.
- **GSE6532** — Affymetrix expression + clinical, from NCBI GEO (used in Lesson 1's batch-effect demo).

*The first run of the METABRIC download takes a few minutes. Subsequent runs are instant.*


In [ ]:
# Data loaders. METABRIC comes from the cBioPortal datahub (now served as individual
# files via GitHub LFS); GSE6532 comes from NCBI GEO over HTTPS. Run once; cached after.
CBIO_BASE = ("https://media.githubusercontent.com/media/cBioPortal/datahub/"
             "master/public/brca_metabric")
CBIO_FILES = {
    "expr":    "data_mrna_illumina_microarray.txt",   # large (~660 MB)
    "patient": "data_clinical_patient.txt",
    "sample":  "data_clinical_sample.txt",
}

def _download(url, dest):
    """Stream a URL to dest, skipping the download if a non-empty cache exists."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    print(f"Downloading {os.path.basename(dest)} ...")
    r = requests.get(url, stream=True, timeout=300, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(dest, "wb") as fh:
        for chunk in r.iter_content(chunk_size=1 << 20):
            fh.write(chunk)
    return dest

def _read_cbio_clinical(path):
    """cBioPortal clinical files carry 4 '#'-commented header lines above the real header."""
    return pd.read_csv(path, sep="\t", comment="#", low_memory=False)

def load_metabric(data_dir=DATA_DIR):
    """Download & parse METABRIC (brca_metabric) from the cBioPortal datahub.

    The expression file is large (~660 MB); the first run takes a few minutes, then caches.

    Returns
    -------
    expr : DataFrame  (genes x samples)  -- Illumina HT-12 microarray, log-intensity
    clin : DataFrame  (samples x clinical fields)
    """
    paths = {k: _download(f"{CBIO_BASE}/{fn}", os.path.join(data_dir, fn))
             for k, fn in CBIO_FILES.items()}
    expr = pd.read_csv(paths["expr"], sep="\t", low_memory=False)
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.dropna(subset=["Hugo_Symbol"]).set_index("Hugo_Symbol")
    expr = expr[~expr.index.duplicated(keep="first")]      # one row per gene symbol
    pat  = _read_cbio_clinical(paths["patient"])
    smp  = _read_cbio_clinical(paths["sample"])
    clin = pat.merge(smp, on="PATIENT_ID", how="inner", suffixes=("", "_smp"))
    if "SAMPLE_ID" in clin.columns:
        clin = clin.set_index("SAMPLE_ID")
    return expr, clin


GSE_SOFT_URL = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6532/"
                "soft/GSE6532_family.soft.gz")

def load_gse6532(data_dir=DATA_DIR):
    """Download & parse GSE6532 (Loi et al.) from GEO.

    We fetch the family SOFT file over HTTPS (more reliable than GEOparse's default FTP),
    restrict to the single most-represented Affymetrix platform, and map its probes to
    gene symbols via the platform (GPL) annotation. The result is a symbol-indexed matrix
    that shares thousands of genes with METABRIC -- the common feature space the PCA
    batch demo needs.

    Returns
    -------
    expr : DataFrame  (genes x samples)  -- Affymetrix, symbol-collapsed (mean over probes)
    clin : DataFrame  (samples x clinical fields)
    """
    import GEOparse, collections
    soft = _download(GSE_SOFT_URL, os.path.join(data_dir, "GSE6532_family.soft.gz"))
    gse  = GEOparse.get_GEO(filepath=soft, silent=True)

    plat_of = {name: gsm.metadata.get("platform_id", ["?"])[0]
               for name, gsm in gse.gsms.items()}
    dom = collections.Counter(plat_of.values()).most_common(1)[0][0]   # dominant platform

    # probe -> gene symbol from the platform annotation
    gpl = gse.gpls[dom].table
    sym_col = next((c for c in gpl.columns
                    if c.lower() in ("gene symbol", "gene_symbol", "symbol")), None)
    pmap = gpl.set_index("ID")[sym_col].dropna().astype(str)
    pmap = pmap[pmap.str.len() > 0]

    cols, meta_rows = {}, {}
    for name, gsm in gse.gsms.items():
        if plat_of[name] != dom:
            continue
        tbl = gsm.table
        if tbl is None or "VALUE" not in tbl.columns:
            continue
        cols[name] = pd.Series(tbl["VALUE"].values, index=tbl["ID_REF"].astype(str).values)
        ch = gsm.metadata.get("characteristics_ch1", [])
        row = {"title": "; ".join(gsm.metadata.get("title", []))}
        for item in ch:
            if ":" in item:
                k, v = item.split(":", 1)
                row[k.strip().lower()] = v.strip()
        meta_rows[name] = row

    expr = pd.DataFrame(cols)                        # probes x samples
    expr = expr[expr.index.isin(pmap.index)]
    expr.index = pmap.loc[expr.index].values         # probes -> gene symbols
    expr = expr.groupby(level=0).mean()              # collapse probes per gene
    clin = pd.DataFrame(meta_rows).T
    return expr, clin

print("Loaders defined: load_metabric(), load_gse6532()")

In [ ]:
# Fetch both cohorts into the shared cache (cached after first run).
metabric_expr, metabric_clin = load_metabric()
print("METABRIC expression (genes x samples):", metabric_expr.shape)
print("METABRIC clinical:", metabric_clin.shape)

gse_expr, gse_clin = load_gse6532()
print("GSE6532 expression (probes->symbols x samples):", gse_expr.shape)
print("GSE6532 clinical:", gse_clin.shape)


## Section 4 — Verify the cache

A final check: list what now lives in the cache and assert the two headline files are present. If this cell prints **“You are ready”**, every lesson notebook will load its data instantly.


In [ ]:
# List the cached files and their sizes, then assert the headline files are present.
import os
data_dir = os.path.abspath(DATA_DIR)
print("Shared cache:", data_dir, "\n")

total = 0
for f in sorted(os.listdir(data_dir)):
    p = os.path.join(data_dir, f)
    if os.path.isfile(p):
        sz = os.path.getsize(p); total += sz
        print(f"  {f:45s} {sz/1e6:9.1f} MB")
print(f"  {'-'*45} {'-'*9}")
print(f"  {'TOTAL':45s} {total/1e6:9.1f} MB\n")

assert any("illumina_microarray" in f for f in os.listdir(data_dir)), "METABRIC expression file missing!"
assert any("GSE6532" in f for f in os.listdir(data_dir)),             "GSE6532 SOFT file missing!"
print("All expected files present. You are ready for Lessons 1-5 — their download cells will hit this cache.")


## What next?

You're set up. From here:

1. Open **`lessons/lesson01_biological_question/practical/task/notebooks/lesson01_exercises.ipynb`**
   (make sure the **`Python (ml26)`** kernel is selected). Lesson 1 loads the data straight from this
   shared cache, reveals the loader functions, and **saves a prepared-cohort checkpoint** that the
   later lessons load.
2. Read each lesson's **Student Pack** (`theory/notes/Lecture N - Student Pack.md`) before its practical.

*You do not need to re-run this notebook unless you delete the cache or move the repository.*
